In [1]:
import numpy as np
import pandas as pd

In [2]:
transactions = pd.DataFrame({
    "transaction_id": [1001,1002,1003,1004,1005,1006,1007,1008,1009,1010],
    "customer_id": ["C101","C102","C103","C101","C104","C105","C106","C107","C108","C109"],
    "transaction_time": [
        "2026-09-20 10:15:00",
        "2026-09-20 11:30:00",
        "2026-09-20 12:45:00",
        "2026-09-20 14:10:00",
        "2026-09-20 15:20:00",
        "2026-09-20 16:05:00",
        "2026-09-20 18:30:00",
        "2026-09-20 20:15:00",
        "2026-09-20 22:40:00",
        "2026-09-20 23:10:00"
    ],
    "amount": [1200,85000,45000,125000,700,60000,95000,15000,55000,200000],
    "payment_method": ["UPI","CARD","CARD","UPI","CASH","NETBANKING","CARD","UPI","CARD","NETBANKING"],
    "country": ["IN","IN","US","SG","IN","IN","UK","IN","US","AE"],
    "device_status": ["TRUSTED","NEW","NEW","NEW","TRUSTED","TRUSTED","NEW","NEW","TRUSTED","NEW"],
    "status": ["SUCCESS","SUCCESS","SUCCESS","SUCCESS","SUCCESS","FAILED","SUCCESS","SUCCESS","SUCCESS","SUCCESS"]
})

In [3]:
transactions.head()

,transaction_id,customer_id,transaction_time,amount,payment_method,country,device_status,status
0,1001,C101,2026-09-20 10:15:00,1200,UPI,IN,TRUSTED,SUCCESS
1,1002,C102,2026-09-20 11:30:00,85000,CARD,IN,NEW,SUCCESS
2,1003,C103,2026-09-20 12:45:00,45000,CARD,US,NEW,SUCCESS
3,1004,C101,2026-09-20 14:10:00,125000,UPI,SG,NEW,SUCCESS
4,1005,C104,2026-09-20 15:20:00,700,CASH,IN,TRUSTED,SUCCESS


In [4]:
transactions.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   transaction_id    10 non-null     int64
 1   customer_id       10 non-null     str  
 2   transaction_time  10 non-null     str  
 3   amount            10 non-null     int64
 4   payment_method    10 non-null     str  
 5   country           10 non-null     str  
 6   device_status     10 non-null     str  
 7   status            10 non-null     str  
dtypes: int64(2), str(6)
memory usage: 772.0 bytes


In [5]:
transactions[transactions.duplicated()]

,transaction_id,customer_id,transaction_time,amount,payment_method,country,device_status,status


high_value_flag — "Y" when amount >= 50000, otherwise "N".

In [6]:
device_risk = {
    'TRUSTED': 'LOW',
    'NEW': 'HIGH'
}

In [7]:
high_value = transactions['amount'] >= 50000
foreign_flag = transactions['country'] != 'IN'
new_status = transactions['device_status'] == 'NEW'

In [11]:
transactions = transactions.assign(
    high_value_flag=np.where(
        transactions['amount'] >= 50000,
        'Y',
        'N'
    ),
    international_flag=np.where(
        transactions['country'] != 'IN',
        'Y',
        'N'
    ),
    device_risk=transactions['device_status'].map(device_risk),
    transaction_fee=transactions['amount'] * 0.02,
    fraud_review_flag=np.where(
        (transactions['status'] == 'SUCCESS') &
        ((high_value & foreign_flag) |
        (high_value & new_status) |
        (foreign_flag & new_status)),
        'REVIEW',
        'CLEAR'
    )
)

In [12]:
transactions

,transaction_id,customer_id,transaction_time,amount,payment_method,country,device_status,status,high_value_flag,international_flag,device_risk,transaction_fee,fraud_review_flag
0,1001,C101,2026-09-20 10:15:00,1200,UPI,IN,TRUSTED,SUCCESS,N,N,LOW,24.0,CLEAR
1,1002,C102,2026-09-20 11:30:00,85000,CARD,IN,NEW,SUCCESS,Y,N,HIGH,1700.0,REVIEW
2,1003,C103,2026-09-20 12:45:00,45000,CARD,US,NEW,SUCCESS,N,Y,HIGH,900.0,REVIEW
3,1004,C101,2026-09-20 14:10:00,125000,UPI,SG,NEW,SUCCESS,Y,Y,HIGH,2500.0,REVIEW
4,1005,C104,2026-09-20 15:20:00,700,CASH,IN,TRUSTED,SUCCESS,N,N,LOW,14.0,CLEAR
5,1006,C105,2026-09-20 16:05:00,60000,NETBANKING,IN,TRUSTED,FAILED,Y,N,LOW,1200.0,CLEAR
6,1007,C106,2026-09-20 18:30:00,95000,CARD,UK,NEW,SUCCESS,Y,Y,HIGH,1900.0,REVIEW
7,1008,C107,2026-09-20 20:15:00,15000,UPI,IN,NEW,SUCCESS,N,N,HIGH,300.0,CLEAR
8,1009,C108,2026-09-20 22:40:00,55000,CARD,US,TRUSTED,SUCCESS,Y,Y,LOW,1100.0,REVIEW
9,1010,C109,2026-09-20 23:10:00,200000,NETBANKING,AE,NEW,SUCCESS,Y,Y,HIGH,4000.0,REVIEW


In [10]:
transactions.loc[
    :,
    [
        'transaction_id', 
        'customer_id',
        'transaction_time',
        'amount',
        'high_value_flag',
        'international_flag',
        'device_risk',
        'transaction_fee',
        'fraud_review_flag'
    ]
]

,transaction_id,customer_id,transaction_time,amount,high_value_flag,international_flag,device_risk,transaction_fee,fraud_review_flag
0,1001,C101,2026-09-20 10:15:00,1200,N,N,LOW,24.0,CLEAR
1,1002,C102,2026-09-20 11:30:00,85000,Y,N,HIGH,1700.0,REVIEW
2,1003,C103,2026-09-20 12:45:00,45000,N,Y,HIGH,900.0,REVIEW
3,1004,C101,2026-09-20 14:10:00,125000,Y,Y,HIGH,2500.0,REVIEW
4,1005,C104,2026-09-20 15:20:00,700,N,N,LOW,14.0,CLEAR
5,1006,C105,2026-09-20 16:05:00,60000,Y,N,LOW,1200.0,CLEAR
6,1007,C106,2026-09-20 18:30:00,95000,Y,Y,HIGH,1900.0,REVIEW
7,1008,C107,2026-09-20 20:15:00,15000,N,N,HIGH,300.0,CLEAR
8,1009,C108,2026-09-20 22:40:00,55000,Y,Y,LOW,1100.0,REVIEW
9,1010,C109,2026-09-20 23:10:00,200000,Y,Y,HIGH,4000.0,REVIEW
